# Gold - Modelo estrella incremental en Parquet para Power BI

Este notebook lee los Parquet normalizados de `data/silver/trip_data_normalized`, procesa solo las fuentes nuevas o modificadas y publica el modelo estrella en `data/gold/modelo_estrella`.

El modelo final contiene una sola `dim_tiempo`, dimensiones descriptivas y `fact_viajes_agregados`. MongoDB no es necesario para ejecutar este flujo.

La incrementalidad se controla con un manifiesto y contribuciones intermedias por fuente. Si una fuente cambia, su contribucion anterior se reemplaza antes de consolidar Gold, evitando sumar dos veces los mismos viajes.

## 1. Entorno y rutas

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import os
import re
import shutil
import subprocess
import sys
import urllib.request


def find_project_root(start=None) -> Path:
    current = Path(start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        normalized = candidate / "data" / "silver" / "trip_data_normalized"
        lookup = candidate / "data" / "lookup" / "taxi_zone_lookup"
        if normalized.exists() and lookup.exists():
            return candidate
    raise FileNotFoundError(
        "No se encontro la raiz que contiene data/silver/trip_data_normalized y data/lookup/taxi_zone_lookup."
    )


PROJECT_ROOT = find_project_root()
NORMALIZED_BASE_DIR = PROJECT_ROOT / "data" / "silver" / "trip_data_normalized"
LOOKUP_DIR = PROJECT_ROOT / "data" / "lookup" / "taxi_zone_lookup"

GOLD_ROOT = PROJECT_ROOT / "data" / "gold" / "modelo_estrella"
DIMENSIONS_DIR = GOLD_ROOT / "dimensiones"
FACTS_DIR = GOLD_ROOT / "hechos"
FACT_OUTPUT_DIR = FACTS_DIR / "fact_viajes_agregados.parquet"
CONTROL_DIR = GOLD_ROOT / "_control"
STAGING_SOURCE_DIR = GOLD_ROOT / "_staging" / "sources"
MANIFEST_PATH = CONTROL_DIR / "process_manifest.json"

LOG_DIR = PROJECT_ROOT / "data" / "logs"
AUDIT_LOG_PATH = LOG_DIR / "gold_parquet_audit.jsonl"

for directory in [DIMENSIONS_DIR, FACTS_DIR, CONTROL_DIR, STAGING_SOURCE_DIR, LOG_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

# Spark en Windows requiere winutils.exe y hadoop.dll.
SYSTEM_HADOOP_HOME = Path("C:/hadoop")
LOCAL_HADOOP_HOME = (
    SYSTEM_HADOOP_HOME
    if SYSTEM_HADOOP_HOME.exists()
    else (PROJECT_ROOT / ".hadoop").resolve()
)
HADOOP_HOME_JAVA = LOCAL_HADOOP_HOME.as_posix()
LOCAL_HADOOP_BIN = LOCAL_HADOOP_HOME / "bin"
LOCAL_HADOOP_BIN.mkdir(parents=True, exist_ok=True)

WINUTILS_BASE_URL = "https://raw.githubusercontent.com/steveloughran/winutils/master/hadoop-3.0.0/bin/"
for hadoop_file in ["winutils.exe", "hadoop.dll"]:
    target = LOCAL_HADOOP_BIN / hadoop_file
    if not target.exists():
        print(f"Descargando {hadoop_file} para PySpark en Windows...")
        urllib.request.urlretrieve(WINUTILS_BASE_URL + hadoop_file, target)

os.environ["HADOOP_HOME"] = HADOOP_HOME_JAVA
os.environ["hadoop.home.dir"] = HADOOP_HOME_JAVA
os.environ["PATH"] = str(LOCAL_HADOOP_BIN) + os.pathsep + os.environ.get("PATH", "")


def java_major_version():
    try:
        result = subprocess.run(["java", "-version"], capture_output=True, text=True, check=False)
        version_text = result.stderr or result.stdout
    except Exception:
        return None
    match = re.search(r'version "(\d+)', version_text)
    return int(match.group(1)) if match else None


def find_jdk17_home():
    for root in [Path("C:/Program Files/Eclipse Adoptium"), Path("C:/Program Files/Java")]:
        if root.exists():
            for candidate in root.glob("**/jdk-17*"):
                if (candidate / "bin" / "java.exe").exists():
                    return candidate
    return None


current_java = java_major_version()
if current_java is not None and current_java > 17:
    jdk17_home = find_jdk17_home()
    if jdk17_home is None:
        raise RuntimeError("Instala JDK 17 para ejecutar PySpark correctamente.")
    os.environ["JAVA_HOME"] = str(jdk17_home)
    os.environ["PATH"] = str(jdk17_home / "bin") + os.pathsep + os.environ.get("PATH", "")

print(f"Proyecto: {PROJECT_ROOT}")
print(f"Silver normalizado: {NORMALIZED_BASE_DIR}")
print(f"Salida Gold: {GOLD_ROOT}")
print(f"Manifiesto: {MANIFEST_PATH}")

Proyecto: C:\Users\mikel\OneDrive\Documentos\ING. SISTEMA\CICLO VII\GESTIÓN DE DATOS MASIVOS\FINAL\Final-Gesti-n_De_Datos_Masivos
Silver normalizado: C:\Users\mikel\OneDrive\Documentos\ING. SISTEMA\CICLO VII\GESTIÓN DE DATOS MASIVOS\FINAL\Final-Gesti-n_De_Datos_Masivos\data\silver\trip_data_normalized
Salida Gold: C:\Users\mikel\OneDrive\Documentos\ING. SISTEMA\CICLO VII\GESTIÓN DE DATOS MASIVOS\FINAL\Final-Gesti-n_De_Datos_Masivos\data\gold\modelo_estrella
Manifiesto: C:\Users\mikel\OneDrive\Documentos\ING. SISTEMA\CICLO VII\GESTIÓN DE DATOS MASIVOS\FINAL\Final-Gesti-n_De_Datos_Masivos\data\gold\modelo_estrella\_control\process_manifest.json


In [2]:
from pyspark import StorageLevel
from pyspark.sql import DataFrame, SparkSession
from pyspark.sql import functions as F

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["PYSPARK_SUBMIT_ARGS"] = (
    "--driver-memory 4g "
    f"--conf spark.driver.extraJavaOptions=-Dhadoop.home.dir={HADOOP_HOME_JAVA} "
    f"--conf spark.executor.extraJavaOptions=-Dhadoop.home.dir={HADOOP_HOME_JAVA} "
    "pyspark-shell"
)

spark = (
    SparkSession.builder
    .master("local[2]")
    .appName("Gold_Parquet_Incremental_TLC")
    .config("spark.ui.enabled", "false")
    .config("spark.driver.memory", "4g")
    .config("spark.executor.memory", "2g")
    .config("spark.driver.maxResultSize", "512m")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.shuffle.partitions", "128")
    .config("spark.sql.files.maxPartitionBytes", str(64 * 1024 * 1024))
    .config("spark.sql.parquet.columnarReaderBatchSize", "1024")
    .config("spark.driver.extraJavaOptions", f"-Dhadoop.home.dir={HADOOP_HOME_JAVA}")
    .config("spark.executor.extraJavaOptions", f"-Dhadoop.home.dir={HADOOP_HOME_JAVA}")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
spark.sparkContext._jvm.java.lang.System.setProperty("hadoop.home.dir", HADOOP_HOME_JAVA)
print("Spark master:", spark.sparkContext.master)
spark

Spark master: local[2]


## 2. Parametros de ejecucion

In [3]:
# False: procesa solo fuentes nuevas o modificadas.
FORCE_REPROCESS_ALL = False

# True: vuelve a publicar hechos y dimensiones desde staging aunque Silver no haya cambiado.
# Util para aplicar cambios de logica o de catalogos sin releer los viajes crudos.
FORCE_REBUILD_OUTPUTS = False

# Mantiene pocos archivos fisicos para que Power BI pueda leerlos con facilidad.
STAGE_FACT_PARTITIONS = 4
FACT_OUTPUT_PARTITIONS = 16

# Activarlo ejecuta validaciones 1:* completas, pero agrega tiempo de procesamiento.
RUN_FULL_REFERENTIAL_VALIDATION = False

print("FORCE_REPROCESS_ALL:", FORCE_REPROCESS_ALL)
print("FORCE_REBUILD_OUTPUTS:", FORCE_REBUILD_OUTPUTS)

FORCE_REPROCESS_ALL: False
FORCE_REBUILD_OUTPUTS: False


## 3. Catalogos oficiales y granularidad

In [4]:
PAYMENT_ROWS = [
    (-1, "No informado / No aplica"),
    (0, "Flex Fare trip"),
    (1, "Credit card"),
    (2, "Cash"),
    (3, "No charge"),
    (4, "Dispute"),
    (5, "Unknown"),
    (6, "Voided trip"),
]

RATECODE_ROWS = [
    (-1, "No informado / No aplica"),
    (1, "Standard rate"),
    (2, "JFK"),
    (3, "Newark"),
    (4, "Nassau or Westchester"),
    (5, "Negotiated fare"),
    (6, "Group ride"),
    (99, "Null / unknown"),
]

TRIP_TYPE_ROWS = [
    (-1, "No informado / No aplica"),
    (1, "Street-hail"),
    (2, "Dispatch"),
]

SHARED_RIDE_ROWS = [
    (-1, "No aplica"),
    (0, "No compartido / no informado"),
    (1, "Compartido"),
]

TAXI_ROWS = [
    ("desconocido", "Tipo desconocido"),
    ("yellow", "Yellow Taxi"),
    ("green", "Green Taxi / SHL"),
    ("fhv", "For-Hire Vehicle"),
]

VENDOR_LABELS = {
    "1": "Creative Mobile Technologies, LLC",
    "2": "Curb Mobility, LLC",
    "6": "Myle Technologies Inc",
    "7": "Helix",
}

# Una fila de la fact representa esta combinacion; no representa un viaje individual.
FACT_KEYS = [
    "time_id",
    "hora_id",
    "tipo_dataset",
    "pickup_location_id",
    "dropoff_location_id",
    "payment_type",
    "ratecode_id",
    "trip_type",
    "shared_ride_flag",
    "provider_key",
]

SOURCE_METRICS = [
    "passenger_count",
    "distancia_millas",
    "duracion_minutos",
    "monto_base",
    "monto_total",
    "propina",
    "peajes",
    "extra",
    "mta_tax",
    "improvement_surcharge",
    "congestion_surcharge",
    "airport_fee",
    "cbd_congestion_fee",
    "ehail_fee",
]

TOTAL_COLUMNS = {
    "passenger_count": "total_pasajeros",
    "distancia_millas": "total_distancia_millas",
    "duracion_minutos": "total_duracion_minutos",
    "monto_base": "total_monto_base",
    "monto_total": "total_monto_total",
    "propina": "total_propina",
    "peajes": "total_peajes",
    "extra": "total_extra",
    "mta_tax": "total_mta_tax",
    "improvement_surcharge": "total_improvement_surcharge",
    "congestion_surcharge": "total_congestion_surcharge",
    "airport_fee": "total_airport_fee",
    "cbd_congestion_fee": "total_cbd_congestion_fee",
    "ehail_fee": "total_ehail_fee",
}

AVERAGE_COLUMNS = {
    "distancia_millas": "promedio_distancia_millas",
    "duracion_minutos": "promedio_duracion_minutos",
    "monto_total": "promedio_monto_total",
    "propina": "promedio_propina",
}

## 4. Funciones de control incremental

In [5]:
def append_audit(record: dict) -> None:
    payload = {"timestamp_utc": datetime.now(timezone.utc).isoformat(), **record}
    with AUDIT_LOG_PATH.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(payload, ensure_ascii=True) + "\n")


def load_manifest() -> dict:
    if not MANIFEST_PATH.exists():
        return {"version": 1, "sources": {}}
    with MANIFEST_PATH.open("r", encoding="utf-8") as handle:
        manifest = json.load(handle)
    manifest.setdefault("version", 1)
    manifest.setdefault("sources", {})
    return manifest


def save_manifest(manifest: dict) -> None:
    temp_path = MANIFEST_PATH.with_suffix(".json.tmp")
    with temp_path.open("w", encoding="utf-8") as handle:
        json.dump(manifest, handle, indent=2, ensure_ascii=True)
    os.replace(temp_path, MANIFEST_PATH)


def parquet_data_files(path: Path) -> list[Path]:
    candidates = [path] if path.is_file() else list(path.rglob("*.parquet"))
    return sorted(file for file in candidates if file.is_file())


def source_fingerprint(path: Path) -> dict:
    files = parquet_data_files(path)
    if not files:
        raise FileNotFoundError(f"No hay archivos Parquet dentro de {path}")
    digest = hashlib.sha256()
    total_bytes = 0
    latest_mtime_ns = 0
    for file in files:
        stat = file.stat()
        relative = file.relative_to(path) if path.is_dir() else Path(file.name)
        digest.update(relative.as_posix().encode("utf-8"))
        digest.update(str(stat.st_size).encode("ascii"))
        digest.update(str(stat.st_mtime_ns).encode("ascii"))
        total_bytes += stat.st_size
        latest_mtime_ns = max(latest_mtime_ns, stat.st_mtime_ns)
    return {
        "fingerprint": digest.hexdigest(),
        "part_files": len(files),
        "total_bytes": total_bytes,
        "latest_mtime_ns": latest_mtime_ns,
    }


def source_id_for(path: Path) -> str:
    relative = path.relative_to(NORMALIZED_BASE_DIR).as_posix()
    slug = re.sub(r"[^a-zA-Z0-9]+", "_", path.stem).strip("_").lower()
    suffix = hashlib.sha1(relative.encode("utf-8")).hexdigest()[:8]
    return f"{slug}_{suffix}"


def path_is_within(path: Path, parent: Path) -> bool:
    try:
        path.resolve().relative_to(parent.resolve())
        return True
    except ValueError:
        return False


def replace_stage_directory(temp_path: Path, target_path: Path) -> None:
    if not path_is_within(temp_path, STAGING_SOURCE_DIR) or not path_is_within(target_path, STAGING_SOURCE_DIR):
        raise ValueError("La ruta de staging esta fuera del directorio Gold permitido.")
    if target_path.exists():
        shutil.rmtree(target_path)
    temp_path.replace(target_path)


def normalize_columns(df: DataFrame) -> DataFrame:
    renamed = df
    for original in df.columns:
        normalized = re.sub(r"[^a-z0-9]+", "_", original.strip().lower()).strip("_")
        if normalized != original:
            renamed = renamed.withColumnRenamed(original, normalized)
    return renamed


def ensure_columns(df: DataFrame, definitions: dict[str, str]) -> DataFrame:
    result = df
    for name, data_type in definitions.items():
        if name not in result.columns:
            result = result.withColumn(name, F.lit(None).cast(data_type))
        else:
            result = result.withColumn(name, F.col(name).cast(data_type))
    return result


def non_empty_string(name: str):
    value = F.trim(F.col(name).cast("string"))
    return F.when(F.length(value) > 0, value)


def source_identity(path: Path) -> tuple[str, int | None]:
    match = re.match(r"^(yellow|green|fhv)_(\d{4})", path.stem.lower())
    if not match:
        return "desconocido", None
    return match.group(1), int(match.group(2))

## 5. Preparacion y contribucion de cada fuente Silver

In [6]:
REQUIRED_TYPES = {
    "tipo_dataset": "string",
    "anio": "int",
    "pickup_datetime": "timestamp",
    "pickup_month": "int",
    "pickup_hour": "int",
    "pickup_location_id": "int",
    "dropoff_location_id": "int",
    "payment_type": "int",
    "ratecode_id": "int",
    "trip_type": "int",
    "shared_ride_flag": "int",
    "vendor_id": "string",
    "dispatching_base_num": "string",
    "affiliated_base_number": "string",
    **{metric: "double" for metric in SOURCE_METRICS},
}


def prepare_source(df: DataFrame, source_path: Path) -> DataFrame:
    taxi_type, source_year = source_identity(source_path)
    prepared = ensure_columns(normalize_columns(df), REQUIRED_TYPES)

    prepared = prepared.withColumn(
        "tipo_dataset",
        F.coalesce(non_empty_string("tipo_dataset"), F.lit(taxi_type)),
    )
    if source_year is not None:
        prepared = prepared.withColumn("anio", F.coalesce(F.col("anio"), F.lit(source_year)))

    valid_year_month = (
        F.col("anio").between(1900, 2200)
        & F.col("pickup_month").between(1, 12)
    )
    prepared = prepared.withColumn(
        "time_id",
        F.coalesce(
            F.date_format("pickup_datetime", "yyyy-MM"),
            F.when(valid_year_month, F.format_string("%04d-%02d", F.col("anio"), F.col("pickup_month"))),
            F.lit("SIN_FECHA"),
        ),
    )

    hour_value = F.coalesce(F.col("pickup_hour"), F.hour("pickup_datetime"))
    prepared = prepared.withColumn(
        "hora_id",
        F.when(hour_value.between(0, 23), hour_value.cast("int")).otherwise(F.lit(-1)),
    )

    for key in [
        "pickup_location_id",
        "dropoff_location_id",
        "payment_type",
        "ratecode_id",
        "trip_type",
        "shared_ride_flag",
    ]:
        prepared = prepared.withColumn(key, F.coalesce(F.col(key).cast("int"), F.lit(-1)))

    provider_id = F.coalesce(
        non_empty_string("vendor_id"),
        non_empty_string("dispatching_base_num"),
        non_empty_string("affiliated_base_number"),
        F.lit("NO_INFO"),
    )
    prepared = (
        prepared
        .withColumn("provider_id", provider_id)
        .withColumn("provider_key", F.concat_ws("|", F.col("tipo_dataset"), F.col("provider_id")))
    )
    return prepared


def build_fact_contribution(prepared: DataFrame) -> DataFrame:
    aggregations = [F.count(F.lit(1)).cast("long").alias("cantidad_viajes")]
    for source_name, total_name in TOTAL_COLUMNS.items():
        aggregations.append(
            F.sum(F.coalesce(F.col(source_name).cast("double"), F.lit(0.0))).alias(total_name)
        )
    for source_name in AVERAGE_COLUMNS:
        aggregations.append(
            F.count(F.when(F.col(source_name).isNotNull(), F.lit(1))).cast("long").alias(f"_count_{source_name}")
        )
    return prepared.groupBy(*FACT_KEYS).agg(*aggregations)


def stage_source(source_info: dict) -> dict:
    source_path = source_info["path"]
    source_id = source_info["source_id"]
    target_path = STAGING_SOURCE_DIR / source_id
    temp_path = STAGING_SOURCE_DIR / f"_tmp_{source_id}"

    if temp_path.exists():
        shutil.rmtree(temp_path)

    print(f"Procesando {source_path.name}...")
    raw = spark.read.parquet(str(source_path))
    prepared = prepare_source(raw, source_path).persist(StorageLevel.DISK_ONLY)

    try:
        input_rows = prepared.count()
        contribution = build_fact_contribution(prepared)
        (
            contribution
            .repartition(STAGE_FACT_PARTITIONS, "time_id")
            .write.mode("overwrite")
            .parquet(str(temp_path / "fact_contribution.parquet"))
        )

        provider_candidates = (
            prepared
            .select("provider_key", "provider_id", "tipo_dataset")
            .dropDuplicates(["provider_key"])
        )
        provider_candidates.coalesce(1).write.mode("overwrite").parquet(
            str(temp_path / "provider_candidates.parquet")
        )

        replace_stage_directory(temp_path, target_path)
        print(f"  Registros Silver: {input_rows:,}")
        print(f"  Staging: {target_path}")
        return {
            "input_rows": input_rows,
            "stage_path": target_path.relative_to(GOLD_ROOT).as_posix(),
        }
    finally:
        prepared.unpersist()
        if temp_path.exists():
            shutil.rmtree(temp_path)

In [7]:
normalized_sources = sorted(
    path
    for path in NORMALIZED_BASE_DIR.glob("*.parquet")
    if path.is_file() or path.is_dir()
)
if not normalized_sources:
    raise FileNotFoundError(f"No se encontraron entradas .parquet en {NORMALIZED_BASE_DIR}")

manifest = load_manifest()
source_infos = []
pending_sources = []

for source_path in normalized_sources:
    source_id = source_id_for(source_path)
    fingerprint = source_fingerprint(source_path)
    info = {
        "path": source_path,
        "source_id": source_id,
        **fingerprint,
    }
    source_infos.append(info)

    previous = manifest["sources"].get(source_id)
    stage_exists = (STAGING_SOURCE_DIR / source_id / "fact_contribution.parquet").exists()
    changed = previous is None or previous.get("fingerprint") != fingerprint["fingerprint"]
    if FORCE_REPROCESS_ALL or changed or not stage_exists:
        pending_sources.append(info)

print(f"Fuentes Silver encontradas: {len(source_infos)}")
print(f"Fuentes pendientes: {len(pending_sources)}")
for info in pending_sources:
    print(" -", info["path"].name)

Fuentes Silver encontradas: 9
Fuentes pendientes: 9
 - fhv_2023.parquet
 - fhv_2024.parquet
 - fhv_2025.parquet
 - green_2023.parquet
 - green_2024.parquet
 - green_2025.parquet
 - yellow_2023.parquet
 - yellow_2024.parquet
 - yellow_2025.parquet


In [8]:
staged_updates = {}
for info in pending_sources:
    started_at = datetime.now(timezone.utc)
    try:
        stage_stats = stage_source(info)
        staged_updates[info["source_id"]] = {
            "path": info["path"].relative_to(PROJECT_ROOT).as_posix(),
            "fingerprint": info["fingerprint"],
            "part_files": info["part_files"],
            "total_bytes": info["total_bytes"],
            "latest_mtime_ns": info["latest_mtime_ns"],
            "processed_at_utc": datetime.now(timezone.utc).isoformat(),
            **stage_stats,
        }
        append_audit({
            "status": "staged",
            "source": info["path"].name,
            "source_id": info["source_id"],
            "input_rows": stage_stats["input_rows"],
            "duration_seconds": round((datetime.now(timezone.utc) - started_at).total_seconds(), 2),
        })
    except Exception as exc:
        append_audit({
            "status": "error",
            "source": info["path"].name,
            "source_id": info["source_id"],
            "error": repr(exc),
        })
        raise

Procesando fhv_2023.parquet...
  Registros Silver: 15,519,045
  Staging: C:\Users\mikel\OneDrive\Documentos\ING. SISTEMA\CICLO VII\GESTIÓN DE DATOS MASIVOS\FINAL\Final-Gesti-n_De_Datos_Masivos\data\gold\modelo_estrella\_staging\sources\fhv_2023_1cbcf3e9
Procesando fhv_2024.parquet...
  Registros Silver: 17,175,195
  Staging: C:\Users\mikel\OneDrive\Documentos\ING. SISTEMA\CICLO VII\GESTIÓN DE DATOS MASIVOS\FINAL\Final-Gesti-n_De_Datos_Masivos\data\gold\modelo_estrella\_staging\sources\fhv_2024_7f60edb9
Procesando fhv_2025.parquet...
  Registros Silver: 24,440,543
  Staging: C:\Users\mikel\OneDrive\Documentos\ING. SISTEMA\CICLO VII\GESTIÓN DE DATOS MASIVOS\FINAL\Final-Gesti-n_De_Datos_Masivos\data\gold\modelo_estrella\_staging\sources\fhv_2025_e52506a6
Procesando green_2023.parquet...
  Registros Silver: 683,391
  Staging: C:\Users\mikel\OneDrive\Documentos\ING. SISTEMA\CICLO VII\GESTIÓN DE DATOS MASIVOS\FINAL\Final-Gesti-n_De_Datos_Masivos\data\gold\modelo_estrella\_staging\sources\gre

## 6. Consolidacion incremental de la tabla de hechos

In [9]:
stage_fact_paths = sorted(STAGING_SOURCE_DIR.glob("*/fact_contribution.parquet"))
publish_needed = bool(pending_sources) or FORCE_REBUILD_OUTPUTS or not FACT_OUTPUT_DIR.exists()

if not stage_fact_paths:
    raise RuntimeError("No existen contribuciones en staging para construir Gold.")

fact_df = None
fact_count = None

if publish_needed:
    contributions = spark.read.parquet(*[str(path) for path in stage_fact_paths])
    additive_columns = ["cantidad_viajes", *TOTAL_COLUMNS.values()]
    count_columns = [f"_count_{source_name}" for source_name in AVERAGE_COLUMNS]

    fact_df = contributions.groupBy(*FACT_KEYS).agg(
        *[F.sum(F.col(name)).alias(name) for name in [*additive_columns, *count_columns]]
    )

    for source_name, average_name in AVERAGE_COLUMNS.items():
        total_name = TOTAL_COLUMNS[source_name]
        count_name = f"_count_{source_name}"
        fact_df = fact_df.withColumn(
            average_name,
            F.when(F.col(count_name) > 0, F.col(total_name) / F.col(count_name)),
        )

    fact_df = fact_df.select(
        *FACT_KEYS,
        "cantidad_viajes",
        *TOTAL_COLUMNS.values(),
        *AVERAGE_COLUMNS.values(),
    ).persist(StorageLevel.DISK_ONLY)

    fact_count = fact_df.count()
    print(f"Filas consolidadas en fact_viajes_agregados: {fact_count:,}")
    fact_df.show(5, truncate=False)
else:
    print("No hay cambios en Silver. Gold no necesita volver a publicarse.")

Filas consolidadas en fact_viajes_agregados: 13,699,527
+-------+-------+------------+------------------+-------------------+------------+-----------+---------+----------------+------------+---------------+---------------+----------------------+----------------------+------------------+-----------------+-----------------+------------+-----------+-------------+---------------------------+--------------------------+-----------------+------------------------+---------------+-------------------------+-------------------------+--------------------+------------------+
|time_id|hora_id|tipo_dataset|pickup_location_id|dropoff_location_id|payment_type|ratecode_id|trip_type|shared_ride_flag|provider_key|cantidad_viajes|total_pasajeros|total_distancia_millas|total_duracion_minutos|total_monto_base  |total_monto_total|total_propina    |total_peajes|total_extra|total_mta_tax|total_improvement_surcharge|total_congestion_surcharge|total_airport_fee|total_cbd_congestion_fee|total_ehail_fee|promedio_di

## 7. Construccion de dimensiones

In [10]:
def complete_numeric_catalog(fact: DataFrame, key: str, rows: list[tuple], label: str) -> DataFrame:
    catalog = spark.createDataFrame(rows, schema=f"{key} int, {label} string")
    observed = fact.select(F.col(key).cast("int").alias(key)).distinct()
    return (
        observed
        .join(catalog, key, "left")
        .withColumn(label, F.coalesce(F.col(label), F.concat(F.lit("Codigo "), F.col(key))))
        .orderBy(key)
    )


def complete_string_catalog(fact: DataFrame, key: str, rows: list[tuple], label: str) -> DataFrame:
    catalog = spark.createDataFrame(rows, schema=f"{key} string, {label} string")
    observed = fact.select(F.col(key).cast("string").alias(key)).distinct()
    return (
        observed
        .join(catalog, key, "left")
        .withColumn(label, F.coalesce(F.col(label), F.concat(F.lit("Tipo "), F.col(key))))
        .orderBy(key)
    )


def build_time_dimension(fact: DataFrame) -> DataFrame:
    month_names = F.create_map(
        *[item for month, name in [
            (1, "Enero"), (2, "Febrero"), (3, "Marzo"), (4, "Abril"),
            (5, "Mayo"), (6, "Junio"), (7, "Julio"), (8, "Agosto"),
            (9, "Septiembre"), (10, "Octubre"), (11, "Noviembre"), (12, "Diciembre"),
        ] for item in (F.lit(month), F.lit(name))]
    )
    result = fact.select("time_id").distinct()
    valid = F.col("time_id").rlike(r"^\d{4}-(0[1-9]|1[0-2])$")
    result = (
        result
        .withColumn("fecha_inicio_mes", F.when(valid, F.to_date(F.concat(F.col("time_id"), F.lit("-01")))))
        .withColumn("anio", F.when(valid, F.substring("time_id", 1, 4).cast("int")).otherwise(F.lit(-1)))
        .withColumn("mes", F.when(valid, F.substring("time_id", 6, 2).cast("int")).otherwise(F.lit(-1)))
        .withColumn("nombre_mes", F.when(valid, F.element_at(month_names, F.col("mes"))).otherwise(F.lit("Sin fecha")))
        .withColumn("trimestre", F.when(valid, F.ceil(F.col("mes") / F.lit(3.0)).cast("int")).otherwise(F.lit(-1)))
        .withColumn(
            "etiqueta_mes",
            F.when(valid, F.concat_ws(" ", F.col("nombre_mes"), F.col("anio"))).otherwise(F.lit("Sin fecha")),
        )
        .withColumn("orden_mes", F.when(valid, F.col("anio") * 100 + F.col("mes")).otherwise(F.lit(-1)))
        .orderBy("orden_mes")
    )
    return result.select(
        "time_id", "fecha_inicio_mes", "anio", "mes", "nombre_mes", "trimestre", "etiqueta_mes", "orden_mes"
    )


def build_hour_dimension(fact: DataFrame) -> DataFrame:
    observed = fact.select(F.col("hora_id").cast("int").alias("hora_id")).distinct()
    return (
        observed
        .withColumn("hora", F.when(F.col("hora_id") >= 0, F.col("hora_id")))
        .withColumn(
            "franja_horaria",
            F.when(F.col("hora_id") == -1, "Sin hora")
            .when(F.col("hora_id").between(0, 5), "Madrugada")
            .when(F.col("hora_id").between(6, 11), "Manana")
            .when(F.col("hora_id").between(12, 17), "Tarde")
            .otherwise("Noche"),
        )
        .orderBy("hora_id")
    )


def build_zone_dimensions(fact: DataFrame) -> tuple[DataFrame, DataFrame]:
    lookup_files = sorted(LOOKUP_DIR.glob("*.parquet"))
    if not lookup_files:
        raise FileNotFoundError(f"No hay archivos Parquet en {LOOKUP_DIR}")
    lookup = normalize_columns(spark.read.parquet(*[str(path) for path in lookup_files]))
    zone_lookup = (
        lookup
        .select(
            F.col("locationid").cast("int").alias("location_id"),
            F.col("borough").cast("string").alias("borough"),
            F.col("zone").cast("string").alias("zone"),
            F.col("service_zone").cast("string").alias("service_zone"),
        )
        .dropDuplicates(["location_id"])
    )

    pickup = (
        fact.select("pickup_location_id").distinct()
        .join(zone_lookup, F.col("pickup_location_id") == F.col("location_id"), "left")
        .select(
            "pickup_location_id",
            F.coalesce("borough", F.lit("No informado")).alias("pickup_borough"),
            F.coalesce("zone", F.lit("No informado")).alias("pickup_zone"),
            F.coalesce("service_zone", F.lit("No informado")).alias("pickup_service_zone"),
        )
        .orderBy("pickup_location_id")
    )
    dropoff = (
        fact.select("dropoff_location_id").distinct()
        .join(zone_lookup, F.col("dropoff_location_id") == F.col("location_id"), "left")
        .select(
            "dropoff_location_id",
            F.coalesce("borough", F.lit("No informado")).alias("dropoff_borough"),
            F.coalesce("zone", F.lit("No informado")).alias("dropoff_zone"),
            F.coalesce("service_zone", F.lit("No informado")).alias("dropoff_service_zone"),
        )
        .orderBy("dropoff_location_id")
    )
    return pickup, dropoff


def build_provider_dimension() -> DataFrame:
    paths = sorted(STAGING_SOURCE_DIR.glob("*/provider_candidates.parquet"))
    candidates = spark.read.parquet(*[str(path) for path in paths])
    providers = candidates.groupBy("provider_key").agg(
        F.first("provider_id", ignorenulls=True).alias("provider_id"),
        F.first("tipo_dataset", ignorenulls=True).alias("tipo_dataset"),
    )
    vendor_map = F.create_map(
        *[item for key, value in VENDOR_LABELS.items() for item in (F.lit(key), F.lit(value))]
    )
    return (
        providers
        .withColumn(
            "tipo_proveedor",
            F.when(F.col("tipo_dataset") == "fhv", "Base FHV").otherwise("Vendor TLC"),
        )
        .withColumn(
            "proveedor",
            F.when(F.col("provider_id") == "NO_INFO", "No informado")
            .when(F.element_at(vendor_map, F.col("provider_id")).isNotNull(), F.element_at(vendor_map, F.col("provider_id")))
            .when(F.col("tipo_dataset") == "fhv", F.concat(F.lit("Base "), F.col("provider_id")))
            .otherwise(F.concat(F.lit("Vendor "), F.col("provider_id"))),
        )
        .select("provider_key", "provider_id", "tipo_dataset", "tipo_proveedor", "proveedor")
        .orderBy("provider_key")
    )

In [11]:
dimensions = {}
dimension_counts = {}

if publish_needed:
    pickup_zone_df, dropoff_zone_df = build_zone_dimensions(fact_df)

    dimensions = {
        "dim_tiempo": build_time_dimension(fact_df),
        "dim_hora": build_hour_dimension(fact_df),
        "dim_tipo_taxi": complete_string_catalog(fact_df, "tipo_dataset", TAXI_ROWS, "tipo_taxi"),
        "dim_zona_pickup": pickup_zone_df,
        "dim_zona_dropoff": dropoff_zone_df,
        "dim_pago": complete_numeric_catalog(fact_df, "payment_type", PAYMENT_ROWS, "forma_pago"),
        "dim_ratecode": complete_numeric_catalog(fact_df, "ratecode_id", RATECODE_ROWS, "ratecode"),
        "dim_trip_type": complete_numeric_catalog(fact_df, "trip_type", TRIP_TYPE_ROWS, "tipo_viaje"),
        "dim_shared_ride": complete_numeric_catalog(
            fact_df, "shared_ride_flag", SHARED_RIDE_ROWS, "viaje_compartido"
        ),
        "dim_proveedor": build_provider_dimension(),
    }

    dimension_keys = {
        "dim_tiempo": "time_id",
        "dim_hora": "hora_id",
        "dim_tipo_taxi": "tipo_dataset",
        "dim_zona_pickup": "pickup_location_id",
        "dim_zona_dropoff": "dropoff_location_id",
        "dim_pago": "payment_type",
        "dim_ratecode": "ratecode_id",
        "dim_trip_type": "trip_type",
        "dim_shared_ride": "shared_ride_flag",
        "dim_proveedor": "provider_key",
    }

    for name, dimension in dimensions.items():
        key = dimension_keys[name]
        rows = dimension.count()
        unique_rows = dimension.select(key).distinct().count()
        if rows != unique_rows:
            raise ValueError(f"{name} contiene claves duplicadas en {key}: {rows} filas, {unique_rows} claves")
        dimension_counts[name] = rows

    if RUN_FULL_REFERENTIAL_VALIDATION:
        for name, dimension in dimensions.items():
            key = dimension_keys[name]
            missing = fact_df.select(key).distinct().join(dimension.select(key), key, "left_anti").count()
            if missing:
                raise ValueError(f"La fact contiene {missing} claves sin correspondencia en {name}.{key}")

    print("Dimensiones validadas:")
    for name, rows in dimension_counts.items():
        print(f" - {name}: {rows:,} filas")

Dimensiones validadas:
 - dim_tiempo: 47 filas
 - dim_hora: 24 filas
 - dim_tipo_taxi: 3 filas
 - dim_zona_pickup: 262 filas
 - dim_zona_dropoff: 261 filas
 - dim_pago: 6 filas
 - dim_ratecode: 8 filas
 - dim_trip_type: 3 filas
 - dim_shared_ride: 1 filas
 - dim_proveedor: 665 filas


## 8. Publicacion de Gold Parquet y actualizacion del manifiesto

In [12]:
if publish_needed:
    publish_started_at = datetime.now(timezone.utc)

    for name, dimension in dimensions.items():
        output_path = DIMENSIONS_DIR / f"{name}.parquet"
        dimension.coalesce(1).write.mode("overwrite").parquet(str(output_path))
        print(f"Guardado: {output_path}")

    (
        fact_df
        .repartition(FACT_OUTPUT_PARTITIONS, "time_id")
        .write.mode("overwrite")
        .parquet(str(FACT_OUTPUT_DIR))
    )
    print(f"Guardado: {FACT_OUTPUT_DIR}")

    manifest["sources"].update(staged_updates)
    manifest["last_gold_publish_utc"] = datetime.now(timezone.utc).isoformat()
    manifest["fact_rows"] = fact_count
    manifest["dimension_rows"] = dimension_counts
    save_manifest(manifest)

    append_audit({
        "status": "gold_published",
        "processed_sources": [info["path"].name for info in pending_sources],
        "fact_rows": fact_count,
        "dimension_rows": dimension_counts,
        "duration_seconds": round((datetime.now(timezone.utc) - publish_started_at).total_seconds(), 2),
        "gold_root": str(GOLD_ROOT),
    })
    fact_df.unpersist()

    print("\nGold publicado correctamente.")
    print(f"Modelo estrella: {GOLD_ROOT}")
    print(f"Auditoria: {AUDIT_LOG_PATH}")
else:
    print("Ejecucion finalizada sin cambios. El modelo Gold existente se conserva.")

Guardado: C:\Users\mikel\OneDrive\Documentos\ING. SISTEMA\CICLO VII\GESTIÓN DE DATOS MASIVOS\FINAL\Final-Gesti-n_De_Datos_Masivos\data\gold\modelo_estrella\dimensiones\dim_tiempo.parquet
Guardado: C:\Users\mikel\OneDrive\Documentos\ING. SISTEMA\CICLO VII\GESTIÓN DE DATOS MASIVOS\FINAL\Final-Gesti-n_De_Datos_Masivos\data\gold\modelo_estrella\dimensiones\dim_hora.parquet
Guardado: C:\Users\mikel\OneDrive\Documentos\ING. SISTEMA\CICLO VII\GESTIÓN DE DATOS MASIVOS\FINAL\Final-Gesti-n_De_Datos_Masivos\data\gold\modelo_estrella\dimensiones\dim_tipo_taxi.parquet
Guardado: C:\Users\mikel\OneDrive\Documentos\ING. SISTEMA\CICLO VII\GESTIÓN DE DATOS MASIVOS\FINAL\Final-Gesti-n_De_Datos_Masivos\data\gold\modelo_estrella\dimensiones\dim_zona_pickup.parquet
Guardado: C:\Users\mikel\OneDrive\Documentos\ING. SISTEMA\CICLO VII\GESTIÓN DE DATOS MASIVOS\FINAL\Final-Gesti-n_De_Datos_Masivos\data\gold\modelo_estrella\dimensiones\dim_zona_dropoff.parquet
Guardado: C:\Users\mikel\OneDrive\Documentos\ING. SIS

## 9. Relaciones que se crean una sola vez en Power BI

| Dimension | Clave unica | Tabla de hechos | Clave foranea |
|---|---|---|---|
| `dim_tiempo` | `time_id` | `fact_viajes_agregados` | `time_id` |
| `dim_hora` | `hora_id` | `fact_viajes_agregados` | `hora_id` |
| `dim_tipo_taxi` | `tipo_dataset` | `fact_viajes_agregados` | `tipo_dataset` |
| `dim_zona_pickup` | `pickup_location_id` | `fact_viajes_agregados` | `pickup_location_id` |
| `dim_zona_dropoff` | `dropoff_location_id` | `fact_viajes_agregados` | `dropoff_location_id` |
| `dim_pago` | `payment_type` | `fact_viajes_agregados` | `payment_type` |
| `dim_ratecode` | `ratecode_id` | `fact_viajes_agregados` | `ratecode_id` |
| `dim_trip_type` | `trip_type` | `fact_viajes_agregados` | `trip_type` |
| `dim_shared_ride` | `shared_ride_flag` | `fact_viajes_agregados` | `shared_ride_flag` |
| `dim_proveedor` | `provider_key` | `fact_viajes_agregados` | `provider_key` |

Todas las relaciones deben tener cardinalidad **uno a muchos (1:*)** y direccion de filtro simple desde la dimension hacia la tabla de hechos.

Para cargar una tabla producida por Spark, en Power BI usa **Obtener datos > Carpeta**, selecciona exactamente la carpeta de la tabla, filtra los archivos con extension `.parquet` y elige **Combinar y transformar**. No selecciones la raiz `modelo_estrella`, porque contiene staging tecnico.

## 10. Ejecuciones siguientes

1. Coloca el nuevo conjunto normalizado como otra entrada `.parquet` dentro de `data/silver/trip_data_normalized`.
2. Ejecuta todas las celdas del notebook Gold.
3. El manifiesto compara ruta, cantidad de partes, tamanos y fechas de modificacion.
4. Solo las fuentes nuevas o modificadas se vuelven a leer desde Silver.
5. Las contribuciones se consolidan y Gold vuelve a publicar las mismas rutas de tablas.
6. Power BI mantiene las relaciones y actualiza sus datos al ejecutar **Actualizar**.

No cambies los nombres de carpetas o columnas Gold despues de configurar Power BI. Para aplicar una modificacion de catalogos sin releer Silver, establece `FORCE_REBUILD_OUTPUTS = True` durante una ejecucion y luego regresalo a `False`.